# V17T2R6 full-8760 data-quality audit

This notebook independently checks the frozen raw-data hashes, the 31-province non-leap load table, the four prepared cases, annual-energy closure, VRE bounds/time-zone declaration, and within-pair identity. It audits inputs only; no solver result is inferred.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
ROOT = Path.cwd().parent.resolve()
assert (ROOT / 'VERSION_V17T2R6_FULL8760.txt').is_file(), ROOT
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fh:
        for block in iter(lambda: fh.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
ROOT

WindowsPath('C:/Users/吴春雷/Desktop/储能/0727 - 副本/V17T2R6_FULL8760_RELEASE_20260804')

In [2]:
raw = ROOT / 'data/raw/load/figshare_29832701_Data_output.xlsx'
processed = ROOT / 'data/processed/figshare_load_2024_31province_8760_nonleap.csv'
load = pd.read_csv(processed)
load_cols = [c for c in load if c.endswith('_load_gw')]
basic = {
 'raw_figshare_sha256': sha256(raw) == 'a93ff6d4a2f4f8c36538fdc4704e2e88bef16347d217c456a99a7a97d56a19e0',
 'processed_load_sha256': sha256(processed) == '74944cc8d707561ac1b6e3210cfa7035eaa5d0e84a391d42c012de17304291ab',
 'hours_8760': len(load) == 8760 and load.model_hour.tolist() == list(range(1,8761)),
 'province_columns_31': len(load_cols) == 31,
 'all_load_values_positive': load[load_cols].stack().gt(0).all(),
}
pd.DataFrame({'check': basic.keys(), 'pass': basic.values()})

,check,pass
0,raw_figshare_sha256,True
1,processed_load_sha256,True
2,hours_8760,True
3,province_columns_31,True
4,all_load_values_positive,True


In [3]:
scenarios = pd.read_csv(ROOT / 'storage_runner/scenarios_targetyear_linked.csv')
rows = []
for scenario in scenarios.scenario:
    case = ROOT / 'sensitivity_cases' / scenario
    prepared = json.loads((case / 'audits/FULL8760_PREPARED_INPUT_VALIDATION.json').read_text(encoding='utf-8'))
    temporal = json.loads((case / 'audits/FULL8760_TEMPORAL_INPUT_AUDIT.json').read_text(encoding='utf-8'))
    residual = max(abs(float(x['annual_energy_residual_mwh'])) for x in temporal['loads'])
    rows.append({
      'scenario': scenario, 'prepared_status': prepared['status'],
      'timepoints': pd.read_csv(case / 'inputs/timepoints.csv').shape[0],
      'max_zone_annual_energy_residual_mwh': residual,
      'vre_shift_hours': temporal['vre']['utc_to_cst_circular_shift_hours'],
      'vcf_min': temporal['vre']['minimum_cf'], 'vcf_max': temporal['vre']['maximum_cf'],
    })
case_audit = pd.DataFrame(rows)
case_audit

,scenario,prepared_status,timepoints,max_zone_annual_energy_residual_mwh,vre_shift_hours,vcf_min,vcf_max
0,TY2033_FULL8760_OBSERVED,PASS,8760,2.384186e-07,8,0.0,1.0
1,TY2033_FULL8760_OPTIMAL,PASS,8760,2.384186e-07,8,0.0,1.0
2,TY2033_FULL8760_LDES_OBSERVED,PASS,8760,2.384186e-07,8,0.0,1.0
3,TY2033_FULL8760_LDES_OPTIMAL,PASS,8760,2.384186e-07,8,0.0,1.0


In [4]:
pair = json.loads((ROOT / 'audits/V17T2R6_FULL8760_PAIR_IDENTITY_AUDIT.json').read_text(encoding='utf-8'))
gate = json.loads((ROOT / 'audits/PUBLICATION_SOURCE_GATE_CASES.json').read_text(encoding='utf-8'))
final_checks = {
  **basic,
  'four_scenarios': len(scenarios) == 4,
  'all_prepared_pass': case_audit.prepared_status.eq('PASS').all(),
  'all_cases_8760': case_audit.timepoints.eq(8760).all(),
  'annual_energy_closure_0p05_mwh': case_audit.max_zone_annual_energy_residual_mwh.le(0.05).all(),
  'vre_cst_shift_plus_8': case_audit.vre_shift_hours.eq(8).all(),
  'vre_bounded_0_1': case_audit.vcf_min.ge(0).all() and case_audit.vcf_max.le(1).all(),
  'pair_identity_pass': pair['status'] == 'PASS',
  'source_case_gate_pass': gate['status'] == 'PASS' and gate['checks_passed'] == gate['checks_total'],
}
summary = pd.DataFrame({'check': final_checks.keys(), 'pass': final_checks.values()})
assert summary['pass'].all(), summary.loc[~summary['pass']]
summary

,check,pass
0,raw_figshare_sha256,True
1,processed_load_sha256,True
2,hours_8760,True
3,province_columns_31,True
4,all_load_values_positive,True
5,four_scenarios,True
6,all_prepared_pass,True
7,all_cases_8760,True
8,annual_energy_closure_0p05_mwh,True
9,vre_cst_shift_plus_8,True


## Interpretation

All input-level gates pass. The notebook does not assert that the four LPs have been solved. Publication claims must wait for solver-native OPTIMAL status, physical audits, result fingerprints, and the paired full-8760 summary.